In [1]:
from __future__ import annotations

import csv
import json
import math
import random
import re
from dataclasses import dataclass
from datetime import datetime
from pathlib import Path
from typing import Dict, List, Tuple

import numpy as np
import pandas as pd
from sklearn.model_selection import StratifiedKFold
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, confusion_matrix, roc_auc_score

try:
    import torch
    import torch.nn as nn
    import torch.nn.functional as F
except Exception as e:
    raise ImportError("PyTorch is required in this kernel (torch).") from e

try:
    import librosa
except Exception as e:
    raise ImportError("librosa is required in this kernel.") from e

try:
    from torchvision.models import alexnet, AlexNet_Weights
except Exception as e:
    raise ImportError("torchvision is required in this kernel.") from e

from tqdm import tqdm

print("EXPERIMENT8 (VOC-ALS dysarthria) — paper pipeline (rhythmPA + log-mel/delta/delta2 + pretrained AlexNet) + YOUR meta-learning classifier")

# ====================== PATHS ======================
REPO_ROOT = Path("/mnt/d/PhD/ALS_PROJECT1/ALS_Diagnosis_Meta")
VOC_ROOT = REPO_ROOT / "Dataset" / "RUSSIAN"  # your local VOC-ALS folder
VOC_META_XLSX = VOC_ROOT / "VOC-ALS.xlsx"
VOC_RHYTHM_PA = VOC_ROOT / "rhythmPA"

RESULTS_DIR = REPO_ROOT / "Results" / "AL_MODELS" / "exp8_vocals_dysarthria_alexnet_metalearning"
RESULTS_DIR.mkdir(parents=True, exist_ok=True)

FOLD_CSV = RESULTS_DIR / "fold_metrics.csv"
SUMMARY_JSON = RESULTS_DIR / "summary.json"
CACHE_NPZ = RESULTS_DIR / "cache_alexnet_embeddings_rhythmPA.npz"

print("VOC_ROOT:", VOC_ROOT, "| exists:", VOC_ROOT.exists())
print("VOC_META_XLSX:", VOC_META_XLSX, "| exists:", VOC_META_XLSX.exists())
print("VOC_RHYTHM_PA:", VOC_RHYTHM_PA, "| exists:", VOC_RHYTHM_PA.exists())
print("RESULTS_DIR:", RESULTS_DIR)

# ====================== PAPER SETTINGS ======================
N_MELS = 256
HOP_LENGTH = 512
N_FFT = 2048
CENTER = True
POWER = 2.0
EPS = 1e-10

IMG_SIZE = 224
ALEXNET_EMB_DIM = 768
ALEXNET_PRETRAINED = True
ALEXNET_TRAINABLE = False

N_SPLITS = 5
N_REPEATS = 4
BASE_SEED = 42

NUM_EPOCHS = 30
LR = 1e-5

K_SHOT = 3
Q_QUERY = 3
META_BATCH_SIZE = 16

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("device:", DEVICE)


# ====================== VOC-ALS METADATA ======================
# openpyxl is not always installed; provide a minimal .xlsx reader fallback.

def _xlsx_col_to_index(col: str) -> int:
    n = 0
    for ch in col:
        if not ('A' <= ch <= 'Z'):
            break
        n = n * 26 + (ord(ch) - ord('A') + 1)
    return n


def read_xlsx_basic(path: Path) -> pd.DataFrame:
    """Read a simple .xlsx into a DataFrame without openpyxl.

    Supports sharedStrings + sheet1-style numeric/string cells.
    """
    import zipfile
    from xml.etree import ElementTree as ET

    with zipfile.ZipFile(path, 'r') as z:
        # shared strings
        shared = []
        if 'xl/sharedStrings.xml' in z.namelist():
            root = ET.fromstring(z.read('xl/sharedStrings.xml'))
            ns = {'m': 'http://schemas.openxmlformats.org/spreadsheetml/2006/main'}
            for si in root.findall('m:si', ns):
                # concat all <t> in case of rich text
                texts = [t.text or '' for t in si.findall('.//m:t', ns)]
                shared.append(''.join(texts))

        # locate first sheet
        sheet_path = None
        try:
            wb = ET.fromstring(z.read('xl/workbook.xml'))
            ns = {
                'm': 'http://schemas.openxmlformats.org/spreadsheetml/2006/main',
                'r': 'http://schemas.openxmlformats.org/officeDocument/2006/relationships',
            }
            sheet = wb.find('m:sheets/m:sheet', ns)
            rid = sheet.attrib.get('{http://schemas.openxmlformats.org/officeDocument/2006/relationships}id')
            rels = ET.fromstring(z.read('xl/_rels/workbook.xml.rels'))
            nsr = {'r': 'http://schemas.openxmlformats.org/package/2006/relationships'}
            target = None
            for rel in rels.findall('r:Relationship', nsr):
                if rel.attrib.get('Id') == rid:
                    target = rel.attrib.get('Target')
                    break
            if target:
                target = target.lstrip('/')
                if not target.startswith('xl/'):
                    target = 'xl/' + target
                if target in z.namelist():
                    sheet_path = target
        except Exception:
            sheet_path = None

        if sheet_path is None:
            sheet_path = 'xl/worksheets/sheet1.xml'
        if sheet_path not in z.namelist():
            raise FileNotFoundError(f'Could not locate worksheet xml in xlsx (tried {sheet_path})')

        sh = ET.fromstring(z.read(sheet_path))
        ns = {'m': 'http://schemas.openxmlformats.org/spreadsheetml/2006/main'}

        cells = {}
        max_r = 0
        max_c = 0

        for row in sh.findall('.//m:sheetData/m:row', ns):
            for c in row.findall('m:c', ns):
                ref = c.attrib.get('r')
                if not ref:
                    continue
                m = re.match(r'([A-Z]+)(\d+)', ref)
                if not m:
                    continue
                col_letters, row_num = m.group(1), int(m.group(2))
                col_num = _xlsx_col_to_index(col_letters)

                t = c.attrib.get('t')
                v = None
                if t == 'inlineStr':
                    is_el = c.find('m:is', ns)
                    if is_el is not None:
                        texts = [t2.text or '' for t2 in is_el.findall('.//m:t', ns)]
                        v = ''.join(texts)
                else:
                    v_el = c.find('m:v', ns)
                    if v_el is None:
                        v = None
                    else:
                        raw = v_el.text
                        if t == 's':
                            try:
                                v = shared[int(raw)]
                            except Exception:
                                v = raw
                        else:
                            # numeric or general
                            try:
                                if raw is None:
                                    v = None
                                else:
                                    if '.' in raw:
                                        v = float(raw)
                                    else:
                                        v = int(raw)
                            except Exception:
                                v = raw

                cells[(row_num, col_num)] = v
                max_r = max(max_r, row_num)
                max_c = max(max_c, col_num)

        data = [[None for _ in range(max_c)] for _ in range(max_r)]
        for (r, c), v in cells.items():
            data[r - 1][c - 1] = v

        return pd.DataFrame(data)

def load_vocals_xlsx(path: Path) -> pd.DataFrame:
    try:
        df0 = pd.read_excel(path, header=None)
    except Exception as e:
        # pandas may require openpyxl; fall back to a minimal xlsx reader
        if 'openpyxl' in str(e).lower():
            df0 = read_xlsx_basic(path)
        else:
            raise
    header_row = None
    for i in range(min(10, len(df0))):
        if (df0.iloc[i].astype(str).str.strip() == "ID").any():
            header_row = i
            break
    if header_row is None:
        raise ValueError(f"Could not find header row containing 'ID' in: {path}")
    header = df0.iloc[header_row].astype(str).tolist()
    df = df0.iloc[header_row + 1 :].copy()
    df.columns = header
    df = df.reset_index(drop=True)
    return df


def build_dysarthria_labels(df: pd.DataFrame) -> pd.DataFrame:
    need = {"ID", "Category", "ALSFRS-R_SpeechSubscore"}
    missing = [c for c in need if c not in df.columns]
    if missing:
        raise ValueError(f"Missing required columns in VOC-ALS.xlsx: {missing}")

    als = df[df["Category"].astype(str).str.strip().eq("ALS")].copy()
    speech = pd.to_numeric(als["ALSFRS-R_SpeechSubscore"], errors="coerce")
    als = als.assign(_speech=speech)
    als = als.dropna(subset=["_speech"]).copy()

    als["y"] = (als["_speech"].astype(int) < 4).astype(int)
    als = als[["ID", "y", "_speech"]].rename(columns={"_speech": "speech_subscore"})
    return als


def subject_id_from_filename(name: str) -> str:
    """Extract subject ID from VOC-ALS wav filename.

    Expected examples: CT001_rhythmPA.wav (controls), PZ001_rhythmPA.wav (patients).
    """
    m = re.match(r"^((?:CT|PZ)\d+)_", name, flags=re.IGNORECASE)
    if not m:
        raise ValueError(f"Could not parse subject ID from filename: {name}")
    return m.group(1).upper()


df_meta = load_vocals_xlsx(VOC_META_XLSX)
df_als = build_dysarthria_labels(df_meta)

print("ALS subjects in metadata:", len(df_als))
print("Label counts (0=non-dys, 1=dys):", df_als["y"].value_counts().sort_index().to_dict())


# ====================== DISCOVER /pa FILES ======================
wav_paths = sorted([p for p in VOC_RHYTHM_PA.glob("*.wav") if p.is_file()])
if not wav_paths:
    raise FileNotFoundError(f"No .wav files found in: {VOC_RHYTHM_PA}")

rows = []
als_ids = set(df_als["ID"].astype(str).str.strip().str.upper().tolist())
y_by_id = {str(r["ID"]).strip().upper(): int(r["y"]) for _, r in df_als.iterrows()}
speech_by_id = {str(r["ID"]).strip().upper(): int(r["speech_subscore"]) for _, r in df_als.iterrows()}

for wp in wav_paths:
    sid = subject_id_from_filename(wp.name)
    if sid not in als_ids:
        continue
    rows.append({"subject_id": sid, "wav_path": str(wp), "y": y_by_id[sid], "speech": speech_by_id[sid]})

df = pd.DataFrame(rows)
if df.empty:
    raise ValueError("No rhythmPA wavs matched ALS subject IDs from metadata.")

print("Matched ALS rhythmPA wavs:", len(df))
print("Unique ALS subjects with rhythmPA:", df["subject_id"].nunique())
print("Label counts in matched wavs:", df["y"].value_counts().sort_index().to_dict())


# ====================== FEATURE EXTRACTION (paper-aligned) ======================
def wav_to_3ch_logmel_delta(path: str) -> np.ndarray:
    y, sr = librosa.load(path, sr=None, mono=True)
    S = librosa.feature.melspectrogram(
        y=y,
        sr=sr,
        n_mels=N_MELS,
        n_fft=N_FFT,
        hop_length=HOP_LENGTH,
        center=CENTER,
        power=POWER,
    )
    logS = np.log(np.maximum(S, EPS)).astype(np.float32)
    d1 = librosa.feature.delta(logS).astype(np.float32)
    d2 = librosa.feature.delta(logS, order=2).astype(np.float32)
    x = np.stack([logS, d1, d2], axis=0)  # (3, 256, T)

    xt = torch.from_numpy(x)[None, ...]  # (1,3,256,T)
    xt = F.interpolate(xt, size=(IMG_SIZE, IMG_SIZE), mode="bilinear", align_corners=False)
    x = xt.squeeze(0).numpy()
    return x


class AlexNetEmbedder(nn.Module):
    def __init__(self, out_dim: int = 768, pretrained: bool = True, trainable: bool = False):
        super().__init__()
        weights = None
        if pretrained:
            try:
                weights = AlexNet_Weights.DEFAULT
            except Exception:
                weights = None
        try:
            base = alexnet(weights=weights)
        except Exception as e:
            raise RuntimeError(
                "Failed to load pretrained AlexNet weights. If you are offline, you may need to pre-download "
                "the torchvision AlexNet checkpoint into your local cache, or set ALEXNET_PRETRAINED=False."
            ) from e

        self.features = base.features
        self.avgpool = base.avgpool
        self.fc1 = base.classifier[1]  # Linear(9216->4096)
        self.fc2 = base.classifier[4]  # Linear(4096->4096)
        self.proj = nn.Linear(4096, int(out_dim))

        if not trainable:
            for p in self.features.parameters():
                p.requires_grad = False
            for p in self.fc1.parameters():
                p.requires_grad = False
            for p in self.fc2.parameters():
                p.requires_grad = False

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        x = self.features(x)
        x = self.avgpool(x)
        x = torch.flatten(x, 1)
        x = F.relu(self.fc1(x))
        x = F.relu(self.fc2(x))
        x = self.proj(x)
        return x


def extract_alexnet_embeddings(df: pd.DataFrame) -> Tuple[np.ndarray, np.ndarray, np.ndarray]:
    model = AlexNetEmbedder(out_dim=ALEXNET_EMB_DIM, pretrained=ALEXNET_PRETRAINED, trainable=ALEXNET_TRAINABLE).to(DEVICE)
    model.eval()

    mean = torch.tensor([0.485, 0.456, 0.406], device=DEVICE).view(1, 3, 1, 1)
    std = torch.tensor([0.229, 0.224, 0.225], device=DEVICE).view(1, 3, 1, 1)

    X_list = []
    y_list = []
    sid_list = []

    for r in tqdm(df.itertuples(index=False), total=len(df), desc="AlexNet embed"):
        x_np = wav_to_3ch_logmel_delta(r.wav_path)  # (3,224,224)
        x = torch.from_numpy(x_np)[None, ...].to(DEVICE)
        x = (x - mean) / std
        with torch.no_grad():
            emb = model(x).detach().cpu().numpy().astype(np.float32).squeeze(0)
        X_list.append(emb)
        y_list.append(int(r.y))
        sid_list.append(str(r.subject_id))

    X = np.vstack(X_list)
    y = np.asarray(y_list, dtype=np.int64)
    sids = np.asarray(sid_list, dtype=object)
    return X, y, sids


if CACHE_NPZ.exists():
    cache = np.load(CACHE_NPZ, allow_pickle=True)
    X_all = cache["X"].astype(np.float32)
    y_all = cache["y"].astype(np.int64)
    subject_ids = cache["subject_ids"].astype(object)
    wav_list = cache["wav_list"].astype(object)
    print("Loaded cache:", CACHE_NPZ)
else:
    X_all, y_all, subject_ids = extract_alexnet_embeddings(df)
    wav_list = df["wav_path"].astype(str).to_numpy(dtype=object)
    np.savez_compressed(CACHE_NPZ, X=X_all, y=y_all, subject_ids=subject_ids, wav_list=wav_list)
    print("Saved cache:", CACHE_NPZ)

print("X_all:", X_all.shape, "| y_all:", y_all.shape, "| subjects:", len(np.unique(subject_ids)))


# ====================== META-LEARNING CLASSIFIER (your pipeline) ======================
class BalancedTaskGenerator:
    def __init__(self, features: np.ndarray, labels: np.ndarray, n_way: int, k_shot: int, q_query: int):
        self.features = features
        self.labels = labels.astype(int)
        self.n_way = int(n_way)
        self.k_shot = int(k_shot)
        self.q_query = int(q_query)

        self.class_indices: Dict[int, List[int]] = {}
        for idx, lab in enumerate(self.labels):
            self.class_indices.setdefault(int(lab), []).append(int(idx))

        need = self.k_shot + self.q_query
        too_small = {c: len(ix) for c, ix in self.class_indices.items() if len(ix) < need}
        if too_small:
            raise ValueError(f"Not enough samples per class for k_shot+q_query={need}: {too_small}. Reduce k_shot/q_query or change split.")

        self.classes = sorted(self.class_indices.keys())

    def create_task(self):
        selected_classes = random.sample(self.classes, self.n_way)
        support_set = []
        query_set = []

        for cls in selected_classes:
            samples = random.sample(self.class_indices[cls], self.k_shot + self.q_query)
            support_set.extend([(samples[i], cls) for i in range(self.k_shot)])
            query_set.extend([(samples[i], cls) for i in range(self.k_shot, self.k_shot + self.q_query)])

        support_features = torch.stack([torch.from_numpy(self.features[idx]).float() for idx, _ in support_set])
        support_labels = torch.LongTensor([label for _, label in support_set])

        query_features = torch.stack([torch.from_numpy(self.features[idx]).float() for idx, _ in query_set])
        query_labels = torch.LongTensor([label for _, label in query_set])

        return support_features, support_labels, query_features, query_labels


class ClassLSTM(nn.Module):
    def __init__(self, feature_dim: int, context_dim: int):
        super().__init__()
        self.lstm = nn.LSTM(input_size=feature_dim, hidden_size=context_dim, num_layers=1, batch_first=True)
        self.fc = nn.Sequential(nn.Linear(context_dim, feature_dim), nn.Tanh())

    def forward(self, x):
        out, _ = self.lstm(x)
        h = out.mean(dim=1)
        return self.fc(h)


class HyperMetaLearner(nn.Module):
    def __init__(
        self,
        input_dim: int,
        hidden_dim: int,
        feature_dim: int,
        context_dim: int,
        num_classes: int,
        use_metric_learner: bool = True,
        distance_metric: str = "cosine",
        distance_scale_init: float = 10.0,
        dropout_p1: float = 0.3,
        dropout_p2: float = 0.2,
    ):
        super().__init__()
        self.feature_dim = int(feature_dim)
        self.num_classes = int(num_classes)
        self.use_metric_learner = bool(use_metric_learner)
        self.distance_metric = str(distance_metric)

        self.feature_net = nn.Sequential(
            nn.Linear(input_dim, hidden_dim),
            nn.LayerNorm(hidden_dim),
            nn.GELU(),
            nn.Dropout(dropout_p1),
            nn.Linear(hidden_dim, hidden_dim),
            nn.LayerNorm(hidden_dim),
            nn.GELU(),
            nn.Dropout(dropout_p2),
            nn.Linear(hidden_dim, feature_dim),
            nn.LayerNorm(feature_dim),
            nn.Tanh(),
        )

        self.class_lstm = ClassLSTM(feature_dim=feature_dim, context_dim=context_dim)

        self.metric_learner = (
            nn.Sequential(nn.Linear(feature_dim, feature_dim), nn.ReLU(), nn.Linear(feature_dim, feature_dim))
            if self.use_metric_learner
            else nn.Identity()
        )

        self.distance_scale = nn.Parameter(torch.tensor(float(distance_scale_init)))

    def forward(self, x):
        return self.feature_net(x)

    def compute_prototypes(self, support, support_labels, n_way: int):
        device_ = next(self.parameters()).device
        support_features = self.feature_net(support)
        protos = torch.zeros(int(n_way), self.feature_dim, device=device_)
        for cls in range(int(n_way)):
            m = support_labels == cls
            if m.any():
                class_features = support_features[m].unsqueeze(0)
                protos[cls] = self.class_lstm(class_features).squeeze(0)
        return protos

    def compute_distances(self, prototypes, query_features):
        prototypes = prototypes + self.metric_learner(prototypes)
        query_features = query_features + self.metric_learner(query_features)

        if self.distance_metric == "euclidean":
            return torch.cdist(query_features, prototypes, p=2) * self.distance_scale
        if self.distance_metric == "cosine":
            q = F.normalize(query_features, p=2, dim=-1)
            p = F.normalize(prototypes, p=2, dim=-1)
            return (1 - torch.matmul(q, p.T)) * self.distance_scale
        if self.distance_metric == "sqeuclidean":
            q = query_features.unsqueeze(1)
            p = prototypes.unsqueeze(0)
            return ((q - p) ** 2).sum(dim=-1) * self.distance_scale
        raise ValueError(f"Unsupported distance metric: {self.distance_metric}")


def prototypical_loss(model, support, support_labels, query, query_labels, n_way: int):
    device_ = next(model.parameters()).device
    support = support.to(device_)
    query = query.to(device_)
    support_labels = support_labels.to(device_)
    query_labels = query_labels.to(device_)

    q_feat = model(query)
    protos = model.compute_prototypes(support, support_labels, n_way)
    dist = model.compute_distances(protos, q_feat)
    log_p = F.log_softmax(-dist, dim=1)
    loss = F.nll_loss(log_p, query_labels)
    preds = torch.argmax(log_p, dim=1)
    acc = (preds == query_labels).float().mean()
    return loss, acc


def compute_prototypes_from_train(model, X_tr: np.ndarray, y_tr: np.ndarray, n_classes: int, batch_size: int = 256) -> torch.Tensor:
    device_ = next(model.parameters()).device
    model.eval()
    sums = None
    counts = torch.zeros(n_classes, device=device_, dtype=torch.long)
    with torch.no_grad():
        for i in range(0, len(X_tr), batch_size):
            xb = torch.from_numpy(X_tr[i : i + batch_size]).float().to(device_)
            yb = torch.from_numpy(y_tr[i : i + batch_size]).long().to(device_)
            feats = model(xb)
            if sums is None:
                sums = torch.zeros((n_classes, feats.shape[1]), device=device_, dtype=feats.dtype)
            for c in range(n_classes):
                m = (yb == c)
                if m.any():
                    sums[c] += feats[m].sum(dim=0)
                    counts[c] += int(m.sum().item())
    if sums is None or (counts == 0).any():
        raise ValueError("Failed to build prototypes (missing class in train?)")
    return sums / counts.clamp(min=1).unsqueeze(1)


def predict_proba(model, prototypes: torch.Tensor, X: np.ndarray, batch_size: int = 256) -> np.ndarray:
    device_ = next(model.parameters()).device
    model.eval()
    probs = []
    with torch.no_grad():
        for i in range(0, len(X), batch_size):
            xb = torch.from_numpy(X[i : i + batch_size]).float().to(device_)
            feats = model(xb)
            dist = model.compute_distances(prototypes, feats)
            p = F.softmax(-dist, dim=1).detach().cpu().numpy()
            probs.append(p)
    return np.vstack(probs)


def train_meta(model, X_tr: np.ndarray, y_tr: np.ndarray, n_classes: int, seed: int):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)

    opt = torch.optim.AdamW(model.parameters(), lr=LR, weight_decay=0.0)
    gen = BalancedTaskGenerator(X_tr, y_tr, n_way=n_classes, k_shot=K_SHOT, q_query=Q_QUERY)

    for _epoch in range(int(NUM_EPOCHS)):
        model.train()
        for _ in range(int(META_BATCH_SIZE)):
            s_x, s_y, q_x, q_y = gen.create_task()
            opt.zero_grad()
            loss, _ = prototypical_loss(model, s_x, s_y, q_x, q_y, n_way=n_classes)
            loss.backward()
            opt.step()


def normalize_train_test(X_tr: np.ndarray, X_te: np.ndarray) -> Tuple[np.ndarray, np.ndarray]:
    mu = X_tr.mean(axis=0)
    sd = X_tr.std(axis=0) + 1e-8
    return (X_tr - mu) / sd, (X_te - mu) / sd


def specificity_from_cm(cm: np.ndarray) -> float:
    tn, fp, fn, tp = cm.ravel()
    return float(tn / max(1, (tn + fp)))


# ====================== CV RUN (subject-disjoint) ======================
fold_rows = []
n_classes = 2

subject_ids_all = subject_ids.copy().astype(str)
y_all_bin = y_all.copy().astype(int)
X_all_emb = X_all.copy().astype(np.float32)

uniq_subjects = np.unique(subject_ids_all)
subj_y = []
for sid in uniq_subjects:
    ys = y_all_bin[subject_ids_all == sid]
    vals, cnts = np.unique(ys, return_counts=True)
    subj_y.append(int(vals[np.argmax(cnts)]))
subj_y = np.asarray(subj_y, dtype=int)

if len(uniq_subjects) != len(subj_y):
    raise ValueError('Subject label construction failed')

rep_seeds = [BASE_SEED + r for r in range(N_REPEATS)]

for rep_i, rep_seed in enumerate(rep_seeds):
    skf = StratifiedKFold(n_splits=N_SPLITS, shuffle=True, random_state=int(rep_seed))
    for fold_i, (tr_s, te_s) in enumerate(skf.split(uniq_subjects, subj_y)):
        tr_subjects = uniq_subjects[tr_s]
        te_subjects = uniq_subjects[te_s]

        tr_idx = np.where(np.isin(subject_ids_all, tr_subjects))[0]
        te_idx = np.where(np.isin(subject_ids_all, te_subjects))[0]

        fold_seed = int(rep_seed * 100 + fold_i)
        random.seed(fold_seed)
        np.random.seed(fold_seed)
        torch.manual_seed(fold_seed)
        if torch.cuda.is_available():
            torch.cuda.manual_seed_all(fold_seed)

        X_tr, y_tr = X_all_emb[tr_idx], y_all_bin[tr_idx]
        X_te, y_te = X_all_emb[te_idx], y_all_bin[te_idx]

        X_tr, X_te = normalize_train_test(X_tr, X_te)

        # sanity for episode feasibility
        need = int(K_SHOT + Q_QUERY)
        tr_counts = {int(c): int(n) for c, n in zip(*np.unique(y_tr, return_counts=True))}
        if any(n < need for n in tr_counts.values()):
            raise ValueError(f"Fold has too few train samples for k+q={need}: {tr_counts}")

        model = HyperMetaLearner(
            input_dim=int(X_tr.shape[1]),
            hidden_dim=512,
            feature_dim=256,
            context_dim=64,
            num_classes=n_classes,
            use_metric_learner=True,
            distance_metric="cosine",
            distance_scale_init=10.0,
            dropout_p1=0.3,
            dropout_p2=0.2,
        ).to(DEVICE)

        train_meta(model, X_tr, y_tr, n_classes=n_classes, seed=fold_seed)
        protos = compute_prototypes_from_train(model, X_tr, y_tr, n_classes=n_classes)
        proba = predict_proba(model, protos, X_te)
        y_pred = np.argmax(proba, axis=1)

        try:
            auc = float(roc_auc_score(y_te, proba[:, 1]))
        except ValueError:
            auc = float("nan")

        acc = float(accuracy_score(y_te, y_pred))
        prec = float(precision_score(y_te, y_pred, pos_label=1, zero_division=0))
        rec = float(recall_score(y_te, y_pred, pos_label=1, zero_division=0))
        f1 = float(f1_score(y_te, y_pred, pos_label=1, zero_division=0))
        cm = confusion_matrix(y_te, y_pred, labels=[0, 1])
        spec = float(specificity_from_cm(cm))

        fold_rows.append(
            {
                "repeat": int(rep_i),
                "fold": int(fold_i),
                "seed": int(fold_seed),
                "n_train_subjects": int(len(tr_subjects)),
                "n_test_subjects": int(len(te_subjects)),
                "n_train": int(len(tr_idx)),
                "n_test": int(len(te_idx)),
                "acc": acc,
                "auc": auc,
                "precision": prec,
                "recall": rec,
                "f1": f1,
                "specificity": spec,
                "tn": int(cm[0, 0]),
                "fp": int(cm[0, 1]),
                "fn": int(cm[1, 0]),
                "tp": int(cm[1, 1]),
            }
        )

        print(f"rep={rep_i} fold={fold_i} acc={acc:.4f} f1={f1:.4f} auc={auc:.4f} prec={prec:.4f} rec={rec:.4f} spec={spec:.4f}")

df_folds = pd.DataFrame(fold_rows)
df_folds.to_csv(FOLD_CSV, index=False)
print("Saved:", FOLD_CSV)

summary = {
    "task": "VOC-ALS dysarthria recognition within ALS (speech_subscore <4 vs =4)",
    "input": "rhythmPA",
    "features": {
        "logmel": True,
        "delta": True,
        "delta2": True,
        "n_mels": int(N_MELS),
        "hop_length": int(HOP_LENGTH),
        "n_fft": int(N_FFT),
        "img_size": int(IMG_SIZE),
    },
    "alexnet": {
        "pretrained": bool(ALEXNET_PRETRAINED),
        "trainable": bool(ALEXNET_TRAINABLE),
        "emb_dim": int(ALEXNET_EMB_DIM),
    },
    "cv": {
        "n_splits": int(N_SPLITS),
        "n_repeats": int(N_REPEATS),
        "base_seed": int(BASE_SEED),
    },
    "meta_learning": {
        "k_shot": int(K_SHOT),
        "q_query": int(Q_QUERY),
        "meta_batch_size": int(META_BATCH_SIZE),
        "num_epochs": int(NUM_EPOCHS),
        "lr": float(LR),
        "model": {
            "hidden_dim": 512,
            "feature_dim": 256,
            "context_dim": 64,
            "use_metric_learner": True,
            "distance_metric": "cosine",
            "distance_scale_init": 10.0,
            "dropout_p1": 0.3,
            "dropout_p2": 0.2,
        },
    },
    "data": {
        "n_samples": int(len(X_all)),
        "n_subjects": int(len(np.unique(subject_ids))),
        "label_counts": {"non_dys": int((y_all == 0).sum()), "dys": int((y_all == 1).sum())},
        "voc_meta_xlsx": str(VOC_META_XLSX),
        "voc_rhythmPA": str(VOC_RHYTHM_PA),
    },
    "results": {
        "mean": df_folds[["acc", "f1", "auc", "precision", "recall", "specificity"]].mean().to_dict(),
        "std": df_folds[["acc", "f1", "auc", "precision", "recall", "specificity"]].std(ddof=0).to_dict(),
    },
}

with open(SUMMARY_JSON, "w", encoding="utf-8") as f:
    json.dump(summary, f, ensure_ascii=False, indent=2)

print("Saved:", SUMMARY_JSON)
print("\n=== FINAL (mean ± std over folds×repeats) ===")
for k in ["acc", "f1", "auc"]:
    mu = summary["results"]["mean"][k]
    sd = summary["results"]["std"][k]
    print(f"{k:12s}: {mu:.4f} ± {sd:.4f}")

# Paper reports these (kept for completeness)
for k in ["precision", "recall", "specificity"]:
    mu = summary["results"]["mean"][k]
    sd = summary["results"]["std"][k]
    print(f"{k:12s}: {mu:.4f} ± {sd:.4f}")


EXPERIMENT8 (VOC-ALS dysarthria) — paper pipeline (rhythmPA + log-mel/delta/delta2 + pretrained AlexNet) + YOUR meta-learning classifier
VOC_ROOT: /mnt/d/PhD/ALS_PROJECT1/ALS_Diagnosis_Meta/Dataset/RUSSIAN | exists: True
VOC_META_XLSX: /mnt/d/PhD/ALS_PROJECT1/ALS_Diagnosis_Meta/Dataset/RUSSIAN/VOC-ALS.xlsx | exists: True
VOC_RHYTHM_PA: /mnt/d/PhD/ALS_PROJECT1/ALS_Diagnosis_Meta/Dataset/RUSSIAN/rhythmPA | exists: True
RESULTS_DIR: /mnt/d/PhD/ALS_PROJECT1/ALS_Diagnosis_Meta/Results/AL_MODELS/exp8_vocals_dysarthria_alexnet_metalearning
device: cuda
ALS subjects in metadata: 102
Label counts (0=non-dys, 1=dys): {0: 53, 1: 49}
Matched ALS rhythmPA wavs: 102
Unique ALS subjects with rhythmPA: 102
Label counts in matched wavs: {0: 53, 1: 49}
Loaded cache: /mnt/d/PhD/ALS_PROJECT1/ALS_Diagnosis_Meta/Results/AL_MODELS/exp8_vocals_dysarthria_alexnet_metalearning/cache_alexnet_embeddings_rhythmPA.npz
X_all: (102, 768) | y_all: (102,) | subjects: 102
rep=0 fold=0 acc=0.5714 f1=0.5263 auc=0.5818 pre